# Data Mining - Lab 6
## Өгөгдлийн хувиргалт: Нормчлол ба интервалд хуваах

**Dataset:** students_scale.csv  

240 оюутны мэдээлэл: `income` (жилийн орлого, USD), `gpa`, `attendance` (ирц, %), `study_hours` (долоо хоногийн суралцах цаг), `score` (0 ба 100-ийн хооронд), `program`, `passed`, `year`. Эхний гурван мөр нь лекцийн жишээтэй (орлого 12,000 / 73,600 / 98,000, оюутан A, B, C) таарна.

### Lab зорилго
- Min-max, z-score, decimal scaling гурван нормчлолыг гараар болон кодоор бодох
- Масштаб, их утга (outlier) нормчлолд хэрхэн нөлөөлдгийг харах
- Нормчлолын параметрийг зөвхөн train өгөгдлөөс тооцож, data leakage-ээс сэргийлэх
- Ижил өргөн, ижил давтамжтай хуваалт хийж, интервалын хилийн дүрмийг зөв хэрэглэх
- Босго (утгын шатлал) ашиглан тоон feature-ийг categorical болгох

## 0) Dataset унших ба ерөнхий шалгалт

Өмнөх lab-уудын адил **pandas** ашиглана. Энэ lab-д өгөгдөл цэвэр (дутуу утгагүй) тул хувиргалтад төвлөрнө.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:\\Users\\Ideapad 5\\Documents\\LAB\\lab1\\normalization\\students_scale.csv")
print(df.shape)
df.head()

(240, 10)


,id,age,income,gpa,attendance,study_hours,score,program,passed,year
0,1001,28,12000,3.2,85,27,75,Программчлал,yes,2026
1,1002,19,73600,3.8,90,25,81,Программчлал,yes,2026
2,1003,24,98000,3.1,88,28,76,Программчлал,yes,2026
3,1004,18,82300,2.9,77,15,66,Программчлал,yes,2026
4,1005,21,25200,4.0,96,13,84,Программчлал,yes,2026


## 1) Масштабын асуудал

Өөр өөр багана өөр өөр нэгж, утгын муждаа байдаг. Зайд суурилсан алгоритм (k-NN, k-means) том муждаа орсон баганыг давамгайлуулдаг.

### TASK 1: Баганын муж

1. `age`, `income`, `gpa`, `attendance` баганы min ба max утгыг ол.
2. Багана бүрийн муж (`max - min`) тооц.
3. Аль багана хамгийн том мужтай вэ?

In [2]:
cols = ["age", "income", "gpa", "attendance"]

# TODO: багана бүрийн min ба max
summary = df[cols].agg(["min", "max"]).T
summary["range"] = summary["max"] - summary["min"]
summary

,min,max,range
age,18.0,30.0,12.0
income,12000.0,310000.0,298000.0
gpa,2.4,4.0,1.6
attendance,57.0,100.0,43.0


### TASK 2: Зайд орлогын нөлөө

Эхний хоёр оюутан A (мөр 0) ба B (мөр 1)-ийн хоорондох Евклидийн зайг `income`, `gpa`, `attendance` гурван баганаар тооц:

d = √[Σ(xᵢ - yᵢ)²]

1. Зайг тооц.
2. Нийт квадрат зөрүүний хэдэн хувийг орлого эзэлж байна вэ?
3. Энэ зай GPA ба ирцийн ялгааг тусгаж байна уу? (үгээр)

In [3]:
a = df.loc[0, ["income", "gpa", "attendance"]].astype(float)
b = df.loc[1, ["income", "gpa", "attendance"]].astype(float)

# TODO: Евклидийн зай
sq = (a - b) ** 2
d_raw = np.sqrt(sq.sum())

print("зай:", round(d_raw, 2))
print("орлогын хувь:", round(sq["income"] / sq.sum() * 100, 4), "%")

зай: 61600.0
орлогын хувь: 100.0 %


## 2) Статистик хэмжүүр ба их утга

Нормчлолын томьёо mean, min, max, SD дээр суурилдаг. Эдгээр хэмжүүр их утганд хэрхэн хариу үйлдэл үзүүлдгийг эхлээд жижиг жишээн дээр харъя.

### TASK 3: Нэг их утгын нөлөө

Бүхэл багц `[2, 4, 4, 4, 5, 5, 7, 9]` (N = 8, дисперсийг N-д хуваана, `ddof=0`).

1. Mean, median, mode, variance, SD-ийг ол.
2. `9`-ийг `25`-аар сольж таван хэмжүүрийг дахин ол.
3. Аль хэмжүүр өөрчлөгдөөгүй вэ? Яагаад? (үгээр)

Хүлээгдэх хариу: анх 5, 4.5, 4, 4, 2. Сольсны дараа 7, 4.5, 4, 48, 6.9282.

In [4]:
def describe5(s):
    # TODO: ddof=0 ашигла
    return {
        "mean": s.mean(),
        "median": s.median(),
        "mode": s.mode().iloc[0],
        "variance": s.var(ddof=0),
        "sd": s.std(ddof=0),
    }

s1 = pd.Series([2, 4, 4, 4, 5, 5, 7, 9])
s2 = s1.replace(9, 25)

pd.DataFrame({"anh": describe5(s1), "9->25": describe5(s2)}).round(4)

,anh,9->25
mean,5.0,7.0000
median,4.5,4.5000
mode,4.0,4.0000
variance,4.0,48.0000
sd,2.0,6.9282


## 3) Min-Max нормчлол

x' = (x - min) / (max - min) × (new_max - new_min) + new_min

Лекцийн жишээ: min = 12,000, max = 98,000, муж [0, 1], v = 73,600 үед x' = 0.7163.

### TASK 4: min_max функц

1. `min_max(x, mn, mx, new_min, new_max)` функц бич.
2. `max == min` үед (тогтмол багана) хуваарь тэг болно. Энэ тохиолдолд `new_min` буцаа.
3. Лекцийн жишээг шалга: 73,600 нь 0.7163 гарах ёстой.

In [5]:
def min_max(x, mn, mx, new_min=0.0, new_max=1.0):
    # тогтмол багана: хуваарь тэг болохоос сэргийлнэ
    if mx == mn:
        return pd.Series(new_min, index=x.index, dtype=float)
    # TODO: томьёог бич
    return (x - mn) / (mx - mn) * (new_max - new_min) + new_min

test = min_max(pd.Series([12000, 73600, 98000]), 12000, 98000)
print(test.round(4).tolist())

[0.0, 0.7163, 1.0]


### TASK 5: Багана дээр хэрэглэх

1. `gpa` ба `attendance` баганыг [0, 1] мужид хувирга. Шинэ баганы нэр `gpa_mm`, `att_mm`.
2. `attendance`-ийг [-1, 1] мужид хувирга (`att_pm`).
3. Шинэ баганы min, max-ийг шалга.
4. `year` багана тогтмол (бүгд 2026). Түүнд `min_max` хэрэглэвэл юу гарах вэ?

In [6]:
# TODO: параметрийг багана тус бүрээс ав
df["gpa_mm"] = min_max(df["gpa"], df["gpa"].min(), df["gpa"].max())
df["att_mm"] = min_max(df["attendance"], df["attendance"].min(), df["attendance"].max())

# TODO: [-1, 1] муж
df["att_pm"] = min_max(df["attendance"], df["attendance"].min(), df["attendance"].max(), new_min=-1, new_max=1)

df["year_mm"] = min_max(df["year"], df["year"].min(), df["year"].max())

df[["gpa_mm", "att_mm", "att_pm", "year_mm"]].agg(["min", "max"]).round(4)

,gpa_mm,att_mm,att_pm,year_mm
min,0.0,0.0,-1.0,0.0
max,1.0,1.0,1.0,0.0


## 4) Z-score нормчлол

z = (x - μ) / σ

Лекцийн жишээ: μ = 54,000, σ = 16,000, v = 73,600 үед z = 1.225. Энэ lab-д σ-г **бүх багцын** (`ddof=0`) утгаар тооцно. pandas-ийн `std()` нь анхдагчаар `ddof=1` тул болгоомжил.

### TASK 6: z_score функц

1. `z_score(x, mu, sigma)` функц бич. `sigma == 0` үед 0 буцаа.
2. Лекцийн жишээг шалга (1.225).
3. `income` баганад хэрэглэж `income_z` үүсгэ. Mean нь 0, `ddof=0` SD нь 1 гарах ёстой.
4. `ddof=1` ба `ddof=0` SD-ийн харьцааг ол. Яагаад өгөгдөл цөөн үед ялгаа томрох вэ? (үгээр)

In [7]:
def z_score(x, mu, sigma):
    if sigma == 0:
        return pd.Series(0.0, index=x.index)
    # TODO: томьёо
    return (x - mu) / sigma

print(round(float(z_score(pd.Series([73600]), 54000, 16000).iloc[0]), 4))

mu = df["income"].mean()
# TODO: ddof=0
sigma = df["income"].std(ddof=0)
df["income_z"] = z_score(df["income"], mu, sigma)

print("mean:", abs(round(df["income_z"].mean(), 6)))
print("sd (ddof=0):", round(df["income_z"].std(ddof=0), 6))
print("ddof=1 / ddof=0:", round(df["income"].std(ddof=1) / df["income"].std(ddof=0), 4))

1.225
mean: 0.0
sd (ddof=0): 1.0
ddof=1 / ddof=0: 1.0021


## 5) Decimal Scaling

v' = v / 10^j, энд j нь `max|v'| < 1` байх **хамгийн бага** бүхэл тоо.

Хатуу `< 1` нөхцөлөөс болж `max|v| = 100` үед `100 / 10² = 1` нөхцөлийг хангахгүй тул j = 3 болно.

### TASK 7: j-г олох

1. `decimal_j(x)` функц бич. Томьёо: `j = floor(log10(max|x|)) + 1`. Максимум 0 үед 0 буцаа.
2. Лекцийн жишээ `[-986, 0, 917]` үед j = 3, `[100]` үед j = 3, `[98000]` үед j = 5 гарах ёстой.
3. `income` баганы j-г ол. Яагаад лекцийн жишээнээс (j = 5) өөр гарав? (үгээр)
4. Нормчлогдсон `income_dec` баганыг үүсгэж, `max|income_dec| < 1` эсэхийг шалга.

In [8]:
def decimal_j(x):
    m = np.abs(x).max()
    if m == 0:
        return 0
    # TODO: хатуу < 1 нөхцөл
    return int(np.floor(np.log10(m))) + 1

print(decimal_j(pd.Series([-986, 0, 917])), decimal_j(pd.Series([100])), decimal_j(pd.Series([98000])))

j = decimal_j(df["income"])
df["income_dec"] = df["income"] / 10 ** j
print("j:", j, "max|v'|:", df["income_dec"].abs().max().round(4))

3 3 5
j: 6 max|v'|: 0.31


## 6) Нормчлолын арга сонгох

Min-max ба z-score нь туйлын (их) утгад мэдрэмтгий. `income` баганад гурван их утга (185,000, 250,000, 310,000) байгааг ашиглан үүнийг харъя.

### TASK 8: Их утгын нөлөө

1. `income`-ийг бүх өгөгдөл дээр min-max-аар хувирга (`mm_all`).
2. Орлого > 100,000 мөрүүдийг хасаж, дахин min-max хий (`mm_clean`).
3. Хоёр тохиолдолд утгуудын хэдэн хувь нь 0.5-аас бага вэ?
4. Их утгуудын z-score хэд вэ?
5. Үр дүнг тайлбарла: их утга min-max-ийн дийлэнх утгыг яагаад жижиг мужид шахав? (үгээр)

In [9]:
inc = df["income"]

mm_all = min_max(inc, inc.min(), inc.max())

# TODO: их утгыг хас
clean = inc[inc <= 100000]
mm_clean = min_max(clean, clean.min(), clean.max())

print("0.5-аас бага (бүгд):", round((mm_all < 0.5).mean() * 100, 1), "%")
print("0.5-аас бага (их утгагүй):", round((mm_clean < 0.5).mean() * 100, 1), "%")

z_all = z_score(inc, inc.mean(), inc.std(ddof=0))
print("их утгын z:", z_all[inc > 100000].round(2).tolist())

0.5-аас бага (бүгд): 98.8 %
0.5-аас бага (их утгагүй): 52.7 %
их утгын z: [4.68, 7.05, 9.23]


### TASK 9: Нормчлол ба зай

Оюутан A, B, C (мөр 0, 1, 2)-ийн `gpa`, `attendance`-аар зайг raw болон min-max хувиргасны дараа харьцуул.

1. d(A,B), d(A,C)-ийг raw утгаар ол. Лекцийн хариу: 5.0359 ба 3.0017.
2. Баганы min, max-ийг **энэ 3 мөрөөс** авч [0, 1]-д хувирга. Лекцийн хариу: A' = (1/7, 0), B' = (1, 1), C' = (0, 0.6).
3. d(A',B'), d(A',C')-ийг дахин ол. A-д аль нь ойр хэвээр үлдсэн бэ?
4. Яагаад raw зайд ирцийн ялгаа давамгайлж байсан бэ? (үгээр)

In [10]:
pts = df.loc[0:2, ["gpa", "attendance"]].astype(float)

def dist(p, q):
    return float(np.sqrt(((p - q) ** 2).sum()))

A, B, C = pts.iloc[0], pts.iloc[1], pts.iloc[2]
print("raw:", round(dist(A, B), 4), round(dist(A, C), 4))

# TODO: багана тус бүрийн min, max
scaled = (pts - pts.min()) / (pts.max() - pts.min())
A2, B2, C2 = scaled.iloc[0], scaled.iloc[1], scaled.iloc[2]
print("scaled:", round(dist(A2, B2), 4), round(dist(A2, C2), 4))
scaled.round(4)

raw: 5.0359 3.0017
scaled: 1.3171 0.6168


,gpa,attendance
0,0.1429,0.0
1,1.0000,1.0
2,0.0000,0.6


## 7) Train/test ба data leakage

Нормчлолын параметр (min, max, μ, σ) -ийг **зөвхөн train** өгөгдлөөс тооцож, test-д тэр параметрээр хэрэглэнэ. Test-ийг тооцоонд оруулбал шалгалтын мэдээлэл сургалтад нэвтэрнэ.

Лекцийн жишээ: train min = 20, max = 60 үед test-ийн 100 нь (100 - 20) / (60 - 20) = 2 болно. Энэ нь алдаа биш, test утга train мужаас гарсныг харуулна.

### TASK 10: Зөв ба буруу дараалал

1. `df.sample(frac=0.8, random_state=42)`-ээр train, үлдсэнийг test болго.
2. `income`-ийн min, max-ийг **train**-ээс ав. Test-ийг тэр параметрээр хувирга (`test_ok`).
3. Test-ийн хэдэн утга [0, 1] мужаас гарав?
4. Буруу арга: min, max-ийг **бүх** df-ээс авч хувирга (`test_leak`). Мужаас гарсан утга хэд болов?
5. Яагаад буруу арга "илүү сайн" харагдах ч хэрэглэж болохгүй вэ? (үгээр)

In [11]:
# TODO: 80% train
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)

# TODO: параметрийг зөвхөн train-ээс
mn, mx = train["income"].min(), train["income"].max()
test_ok = min_max(test["income"], mn, mx)

# буруу арга: бүх өгөгдлөөс
test_leak = min_max(test["income"], df["income"].min(), df["income"].max())

def out_of_range(s):
    return int(((s < 0) | (s > 1)).sum())

print("train:", len(train), "test:", len(test))
print("зөв арга, мужаас гарсан:", out_of_range(test_ok))
print("буруу арга, мужаас гарсан:", out_of_range(test_leak))
print("лекцийн жишээ:", float(min_max(pd.Series([100]), 20, 60).iloc[0]))

train: 192 test: 48
зөв арга, мужаас гарсан: 1
буруу арга, мужаас гарсан: 0
лекцийн жишээ: 2.0


## 8) Интервалд хуваах (Discretization)

Тоон утгыг интервалын шошгоор солино. Интервал `[a, b)` нь a-г **багтаана**, b-г **багтаахгүй**. Сүүлийн интервал max-ийг багтаана.

Лекцийн жишээ: `4, 8, 15, 21, 21, 24, 25, 28, 34`, k = 3.

### TASK 11: Ижил өргөнтэй хуваалт

w = (max - min) / k

1. Лекцийн 9 утгыг k = 3 бүлэгт хуваа. w ба бүлгийн дугаар (0, 1, 2) -ийг ол. Хүлээгдэх тоо: 2, 3, 4.
2. `24` аль бүлэгт орох вэ?
3. `pd.cut(vals, bins=[4, 14, 24, 34], right=False)` хийвэл `34` юу болох вэ? Яагаад? (үгээр)
4. Сүүлийн хилийг `np.inf` болгож засаад дахин шалга.
5. `score` баганыг k = 5 бүлэгт ижил өргөнтэй хуваа. Бүлэг бүрийн тоог ол.

In [12]:
vals = pd.Series([4, 8, 15, 21, 21, 24, 25, 28, 34])
k = 3
w = (vals.max() - vals.min()) / k

# TODO: бүлгийн дугаар. Сүүлийн утга k-1 бүлэгт орно
idx = ((vals - vals.min()) // w).astype(int).clip(upper=k - 1)
print("w:", w, "бүлэг:", idx.tolist())
print("тоо:", idx.value_counts().sort_index().tolist())

# pd.cut-ийн хилийн алдаа
bad = pd.cut(vals, bins=[4, 14, 24, 34], right=False)
print("NaN болсон:", int(bad.isna().sum()))

# TODO: сүүлийн хилийг np.inf болго
good = pd.cut(vals, bins=[4, 14, 24, np.inf], right=False)
print("засвар:", good.value_counts().sort_index().tolist())

# score, k = 5
s = df["score"]
w5 = (s.max() - s.min()) / 5
sidx = ((s - s.min()) // w5).astype(int).clip(upper=4)
sidx.value_counts().sort_index()

w: 10.0 бүлэг: [0, 0, 1, 1, 1, 2, 2, 2, 2]
тоо: [2, 3, 4]
NaN болсон: 1
засвар: [2, 3, 4]


score
0      7
1     82
2    100
3     48
4      3
Name: count, dtype: int64

### TASK 12: Ижил давтамжтай хуваалт

Эрэмбэлсэн утгуудыг бүлэг бүрт ойролцоо тооны утга орохоор хуваана. `pd.qcut` ашиглана.

1. Лекцийн 9 утгыг `q = 3`-аар хуваа. Хүлээгдэх тоо: 3, 3, 3.
2. `income` баганыг ижил өргөнтэй (k = 4) ба ижил давтамжтай (q = 4) хоёр аргаар хуваа. Бүлэг бүрийн тоог хоёр баганатай хүснэгтэд гарга.
3. Их утгууд байгаа тул аль арга нь бүлгүүдийг тэнцүү хуваав? Яагаад? (үгээр)

In [13]:
# TODO: q = 3
eq_freq = pd.qcut(vals, q=3)
print("лекц:", eq_freq.value_counts().sort_index().tolist())

inc = df["income"]
# TODO: ижил өргөнтэй k = 4: хилийг np.linspace-ээр
edges = np.linspace(inc.min(), inc.max(), 5)
width = pd.cut(inc, bins=edges, include_lowest=True)
freq = pd.qcut(inc, q=4)

pd.DataFrame({
    "ижил өргөн": width.value_counts().sort_index().tolist(),
    "ижил давтамж": freq.value_counts().sort_index().tolist(),
})

лекц: [3, 3, 3]


,ижил өргөн,ижил давтамж
0,233,60
1,4,61
2,1,59
3,2,60


### TASK 13: Bin mean зөөллөлт

Бүлэг бүрийн утгыг тэр бүлгийн **дундажаар** солино (smoothing by bin means).

1. Лекцийн 9 утгыг ижил давтамжтай 3 бүлэгт хувааж, утгыг бүлгийн дундажаар сол. Хүлээгдэх хариу: 9, 9, 9, 22, 22, 22, 29, 29, 29.
2. Лекцийн `4, 8, 15` бүлэг `9, 9, 9` болсон. Энэ нь юугаа алдсан бэ? (үгээр)

In [14]:
sorted_vals = vals.sort_values()
groups = pd.qcut(sorted_vals, q=3, labels=False)

# TODO: бүлэг тус бүрийн дундаж
smoothed = sorted_vals.groupby(groups).transform("mean")
smoothed.tolist()

[9.0, 9.0, 9.0, 22.0, 22.0, 22.0, 29.0, 29.0, 29.0]

## 9) Утгын шатлал ба босго

Домэйны мэдлэгээр босго тогтоож, тоон feature-ийг categorical болгоно. Лекцийн босго:

- `[0, 60)` бол **Бага**
- `[60, 80)` бол **Дунд**
- `[80, 100]` бол **Өндөр**

Босгыг өөрчилж болно. Гол нь тухайн dataset-д тохирох утгыг олох.

### TASK 14: score_level

1. `np.select` ашиглан `score_level` багана үүсгэ.
2. Хилийн утгуудыг (мөр 10 ба 14 хүртэл: 60, 80, 100, 59, 79) шалга. 60 нь Дунд, 80 нь Өндөр, 100 нь Өндөр, 59 нь Бага, 79 нь Дунд гарах ёстой.
3. Шатлал бүрийн тоог ол.
4. `pd.cut(df["score"], [0, 60, 80, 100], right=False)` хийвэл 100 юу болох вэ? (үгээр)

In [15]:
# TODO: босгууд
conds = [df["score"] < 60, df["score"] < 80]
df["score_level"] = np.select(conds, ["Бага", "Дунд"], default="Өндөр")

print(df.loc[10:14, ["score", "score_level"]])
print(df["score_level"].value_counts().to_dict())
print("pd.cut-ийн NaN:", int(pd.cut(df["score"], [0, 60, 80, 100], right=False).isna().sum()))

    score score_level
10     60        Дунд
11     80       Өндөр
12    100       Өндөр
13     59        Бага
14     79        Дунд


{'Дунд': 182, 'Өндөр': 40, 'Бага': 18}
pd.cut-ийн NaN: 1


### TASK 15: Босгыг хэрхэн сонгох вэ

Зорилтот class ашиглах (supervised) ба ашиглахгүй (unsupervised) аргыг харьцуул.

Зорилт: `high = score >= 80`. Feature: `attendance`. Ирцийн нэг босго `t` сонгоод `attendance >= t` бол high гэж таамаглая.

1. **Ижил өргөн:** босго = (min + max) / 2.
2. **Ижил давтамж:** босго = median.
3. **Supervised:** ирцийн өвөрмөц утгуудын дундах цэг бүрийг босго болгон туршиж, accuracy хамгийн өндөр босгыг ол.
4. Гурван босгын accuracy-г харьцуул. Яагаад supervised нь хамгийн багадаа бусдаас дор биш байх вэ? (үгээр)

In [16]:
y = (df["score"] >= 80)
att = df["attendance"]

def accuracy(t):
    return float(((att >= t) == y).mean())

t_width = (att.min() + att.max()) / 2
t_freq = att.median()

# TODO: supervised, өвөрмөц утгуудын дундах цэгүүд
u = np.sort(att.unique())
cands = (u[:-1] + u[1:]) / 2
best_t = max(cands, key=accuracy)

print("ижил өргөн:", t_width, round(accuracy(t_width), 3))
print("ижил давтамж:", t_freq, round(accuracy(t_freq), 3))
print("supervised:", best_t, round(accuracy(best_t), 3))

ижил өргөн: 78.5 0.421
ижил давтамж: 83.0 0.621
supervised: 95.5 0.879


## 10) Бусад хувиргалт

Нормчлол ба интервалд хуваахаас гадна шинэ шинж үүсгэх, түүвэрлэх зэрэг хувиргалт бий.

### TASK 16: Түүвэрлэлт ба шинэ шинж

1. Шинэ шинж: `gpa_pct = gpa / 4 * 100`.
2. `program` бүрээс 20%-ийг санамсаргүй авч (`groupby(...).sample`) давхарга бүрээс түүвэрл.
3. Бүх өгөгдөл ба түүвэр дэх `program`-ийн харьцааг (`value_counts(normalize=True)`) харьцуул. Түүвэр төлөөлөх чадвартай юу? (үгээр)

In [17]:
df["gpa_pct"] = df["gpa"] / 4 * 100

# TODO: давхарга бүрээс 20%
sample = df.groupby("program").sample(frac=0.2, random_state=1)

pd.DataFrame({
    "бүх өгөгдөл": df["program"].value_counts(normalize=True).round(3),
    "түүвэр": sample["program"].value_counts(normalize=True).round(3),
})

,бүх өгөгдөл,түүвэр
program,,
Программчлал,0.388,0.396
Сүлжээ,0.358,0.354
Дизайн,0.254,0.250


## 11) Бүх алхмыг нэг дор

### TASK 17: Хувиргалтын pipeline

1. `fit_params(train)` функц нь **зөвхөн train**-ээс `income`-ийн min, max болон `attendance`-ийн μ, σ (`ddof=0`) -г хадгалсан dictionary буцаана.
2. `transform(d, p)` функц нь `income_mm`, `att_z`, `score_level` гурван багана нэмнэ.
3. Train дээр `income_mm`-ийн min = 0, max = 1 гарсныг шалга. Test дээр хэд гарав?

In [18]:
def fit_params(train):
    # TODO: зөвхөн train-ээс
    return {
        "mn": train["income"].min(),
        "mx": train["income"].max(),
        "mu": train["attendance"].mean(),
        "sd": train["attendance"].std(ddof=0),
    }

def transform(d, p):
    out = d.copy()
    out["income_mm"] = min_max(out["income"], p["mn"], p["mx"])
    out["att_z"] = z_score(out["attendance"], p["mu"], p["sd"])
    out["score_level"] = np.select([out["score"] < 60, out["score"] < 80], ["Бага", "Дунд"], default="Өндөр")
    return out

params = fit_params(train)
train_t = transform(train, params)
test_t = transform(test, params)

print("train income_mm:", train_t["income_mm"].min(), train_t["income_mm"].max())
print("test income_mm:", round(test_t["income_mm"].min(), 4), round(test_t["income_mm"].max(), 4))
print("train att_z mean:", abs(round(train_t["att_z"].mean(), 6)))

train income_mm: 0.0 1.0
test income_mm: 0.0458 1.2521
train att_z mean: 0.0


## 12) Дүгнэлт асуултууд (Concept Check)

### TASK 18: Verbal

1. Min-max, z-score, decimal scaling-ийн аль нь нийтлэг муж [0, 1] өгдөг вэ? Аль нь дундажийг 0 болгодог вэ?
2. Орлого 310,000 гэсэн нэг их утга min-max хувиргалтыг яаж өөрчилдөг вэ? Ямар хувиргалтыг илүү найдвартай гэж үзэх вэ?
3. Ижил өргөн ба ижил давтамжтай хуваалтын аль нь их утгатай өгөгдөлд тохиромжтой вэ?
4. `[60, 80)` ба `[60, 80]` интервалын ялгаа юу вэ? Яагаад хилийн дүрмийг заавал тодорхойлдог вэ?
5. Яагаад discretization хийхэд ижил шошготой 60 ба 79 оноо ялгагдахгүй болдог вэ? Энэ нь ямар үед хор болох вэ?

### Тайлбар
- T1: income хамгийн том мужтай (298,000).
- T2: зайн бараг 100% нь орлогоос гарсан, gpa ба ирцийн ялгаа огт харагдахгүй байна.
- T3: median, mode өөрчлөгдөөгүй, учир нь тэд дунд байрлал болон давтамжаас хамаардаг, их утгын хэмжээнээс биш.
- T5: year тогтмол тул бүх утга 0 болсон.
- T6: ddof=1 нь N-1-д хуваадаг. N бага үед N ба N-1 ялгаа их тул SD илүү зөрнө.
- T7: income-ийн max нь 310,000 тул j = 6 гарсан.
- T8: 310,000 гэх мэт их утга max-ийг татаж, ихэнх утга 0-0.3 орчимд шахагдсан.
- T9: хоёр тохиолдолд C нь A-д ойр. Raw үед ирцийн ялгаа (5, 3) gpa-гаас (0.6, 0.1) хамаагүй том учир давамгайлсан.
- T10: буруу арга test-ийн мэдээллийг ашигласан тул мужаас гарсангүй, гэхдээ бодит ажилд test-ийг урьдчилж мэдэхгүй.
- T11: 24 нь 2-р бүлэгт (index 2) орно. pd.cut-д 34 нь [24, 34) -д багтахгүй тул NaN болсон.
- T12: ижил давтамж нь тэнцүү хуваасан, ижил өргөн дээр их утгууд мужийг сунгаж бараг бүгд эхний бүлэгт орсон.
- T13: бүлэг доторх ялгаа алдагдсан (4 ба 15 адилхан 9 болсон).
- T14: pd.cut-д right=False үед 100 нь [80, 100) -д орохгүй, NaN болно.
- T15: supervised нь бүх боломжит босгыг шалгаж хамгийн сайныг авдаг тул бусдаас дор гарахгүй (0.879).
- T16: түүврийн харьцаа бүх өгөгдөлтэй бараг ижил тул төлөөлж чадна.
- T17: test дээр max 1.25 гарсан, train-ээс их орлоготой оюутан test-д орсон.

**TASK 18**
1. Min-max [0, 1] өгдөг, z-score дунджийг 0 болгодог.
2. Max-ийг их болгож бусад утгыг 0 орчимд шахна. Ийм үед z-score эсвэл их утгыг хассан хувиргалт найдвартай.
3. Ижил давтамжтай.
4. `[60, 80)` 80-ийг багтаахгүй, `[60, 80]` багтаана. Хилийн утга хоёр бүлэгт орох эсвэл аль ч бүлэгт орохгүй байхаас сэргийлнэ.
5. Нэг интервалд орсон утгууд нэг шошготой болно. 79 нь 80-тай бараг адил ч өөр шошго авдаг тул нарийн ялгаа чухал үед хортой.


# Data Mining - Lab 6 Тайлан
## Өгөгдлийн хувиргалт: Нормчлол ба интервалд хуваах

**Нэр:**  
**Оюутны код:**  
**Анги / Бүлэг:**  
**Огноо:**  

---

## 1. Lab зорилго
Тоон өгөгдлийг min-max, z-score, decimal scaling аргаар нормчилж, интервалд хувааж сурах. Мөн train/test дээр data leakage гаргахгүй хувиргах.

---

## 2. Dataset-ийн товч тайлбар
- Dataset нэр: students_scale.csv
- Мөр, баганын тоо: 240 мөр, 10 багана
- Ашигласан үндсэн баганууд: income, gpa, attendance, score

240 оюутны орлого, голч дүн, ирц, суралцах цаг, оноо зэрэг мэдээлэл байна. Дутуу утга байхгүй ч income баганад 185,000, 250,000, 310,000 гэсэн их утгууд бий.

---

## 3. Нормчлол

### 3.1 Гурван арга
| Арга | Томьёо | income дээрх үр дүн (мөр 1) |
|---|---|---|
| Min-max | (x - min) / (max - min) | 0.2067 |
| Z-score | (x - μ) / σ | 0.6274 |
| Decimal scaling | x / 10^j, j = 6 | 0.0736 |

### 3.2 Их утгын нөлөө
- Их утгатай үед утгуудын 98.8% нь 0.5-аас бага гарсан. Их утгыг хассаны дараа 52.7% болсон.
- Их утга байгаа үед z-score-ийг сонгоно, учир нь min-max шиг бусад утгыг нэг жижиг хэсэгт шахдаггүй.

---

## 4. Data leakage
- Параметрийг зөвхөн train (192 мөр)-ээс тооцсон.
- Test-ийн 1 утга [0, 1] мужаас гарсан (max 1.25), учир нь test-д train-ий max-аас их орлого байсан.

---

## 5. Интервалд хуваах

| Арга | Бүлгийн тоо (score, k = 5) | Ажигласан зүйл |
|---|---|---|
| Ижил өргөн | 7, 82, 100, 48, 3 | Дунд бүлгүүдэд ихэнх нь орсон |
| Ижил давтамж | 51, 48, 52, 49, 40 | Бараг тэнцүү хуваагдсан |
| Босго (Бага, Дунд, Өндөр) | 18, 182, 40 | Ихэнх нь Дунд |

- `[a, b)` дүрмийг ашигласан, сүүлийн интервалд max-ийг багтаасан (np.inf эсвэл clip).

---

## 6. Босго сонгох
- Ижил өргөн 0.421, ижил давтамж 0.621, supervised 0.879.
- Supervised нь илүү, учир нь зорилтот class-ийг харж бүх босгоос хамгийн сайныг нь сонгодог.

---

## 7. Reflection

1. Нормчлолыг хуваахаас өмнө хийвэл test-ийн мэдээлэл train-д орж, үр дүн бодитоос сайн харагдана.
2. Classification-д зайд суурилсан алгоритм ашиглахаас өмнө өгөгдлийг нормчлох хэрэгтэйг ойлгосон.
3. pd.cut-ийн хилийн дүрэм (right=False үед max NaN болдог) эхэндээ ойлгомжгүй байсан.

---

## 8. Дүгнэлт

Их утгатай өгөгдөлд min-max тохиромжгүй, z-score эсвэл ижил давтамжтай хуваалт илүү тохирно. Нормчлолын параметрийг заавал зөвхөн train-ээс тооцох ёстой.
